# Assignment 01: Data preparation and district analysis

**Overall purpose:** Answer the five crime-data questions on slide 11 using reproducible topic modeling and evidence-based report analysis. Your section must connect to the shared final report table and the main workstation.

## Shared rules: read before writing code

1. **Claim one assignment.** Add your name and reviewer below, then tell the group. These six notebooks are roles, not six tasks for each person.
2. **Use the shared files.** The original Parquet and shared/reports_base.parquet are read-only inputs. The base adds stable report_id values but does not clean or classify anything.
3. **Keep report_id.** Never recreate IDs after filtering, sorting or resetting an index. Join results using report_id with merge validation. URL is not a unique ID: some URLs repeat.
4. **Use one prepared version.** Member 1 must publish outputs/member_01/reports_prepared.parquet before final modeling. Preserve all source rows and mark exclusions. Everyone records that file's checksum in run_metadata.json. Prototype results on the base are exploratory and must be rerun on the agreed prepared file.
5. **Preserve alignment.** Build each text list and embedding matrix from the same filtered DataFrame in the same order. Never filter/sort text and embeddings separately.
6. **Separate analysis from evidence.** Write short code cells, show readable tables/plots, and explain actual outputs. Do not invent findings, silently fill missing districts, or treat a colored cluster plot as proof of meaningful crime types.
7. **Export to your own folder.** outputs/member_XX holds submissions; figures/member_XX holds readable figures. templates/member_XX contains empty examples and is never a finished submission.
8. **Finish with a peer review.** Everyone contributes their section, interpretation and sample reading. Member 6 coordinates validation; they do not write the entire group project.

**Your name:** ______  
**Peer reviewer:** ______  
**Status:** Not started / In progress / Ready for review / Accepted  
**Prepared input version/checksum:** ______


## Setup and input check

Keep this notebook inside team_notebooks, or beside the shared folder. the notebook alone does not contain the dataset.

In [1]:
from pathlib import Path
import pandas as pd
import json

In [2]:
project_folder = Path.cwd()
if not (project_folder / "shared").exists():
    project_folder = project_folder.parent

assert (project_folder / "shared/source_manifest.json").exists(), "Open the notebook from the shared project folder."

In [3]:
output_folder = project_folder / "outputs/member_01"
figure_folder = project_folder / "figures/member_01"
output_folder.mkdir(parents=True, exist_ok=True)
figure_folder.mkdir(parents=True, exist_ok=True)

In [4]:
reports_base = pd.read_parquet(project_folder / "shared/reports_base.parquet")
assert reports_base["report_id"].is_unique
print("Base reports:", len(reports_base))
reports_base[["report_id", "date_posted", "title_en", "district"]].head(3)

Base reports: 5126


,report_id,date_posted,title_en,district
0,BR000000,2026-09-03,Police release wanted photo after shooting in ...,Woltmershausen
1,BR000001,2026-09-03,Burglary in Bar - Police Catch Perpetrators,Ostertor
2,BR000002,2026-09-02,Paint vandalism at high school,Horn


In [5]:
prepared_path = project_folder / "outputs/member_01/reports_prepared.parquet"
if prepared_path.exists():
    reports = pd.read_parquet(prepared_path)
    print("Prepared input available. Check its version before modeling.")
else:
    print("Preparation pending: final modeling and group analysis must wait for Member 1.")

Preparation pending: final modeling and group analysis must wait for Member 1.


### Packages

Use the agreed group environment. From the project folder, the setup command is `python -m pip install -r requirements.txt`. Do this only when preparing the modeling environment. The skeleton uses pandas/pyarrow; BERTopic, UMAP and HDBSCAN are not installed in the environment used to check these starter notebooks. No algorithm has been run yet.

## Things to note throughout the project

- Count reports, not confirmed incidents or individual victims. One report can discuss several events or offenses. Treat the unit consistently.
- date_posted is a publication date. The first/last years and boundary months are partial; the source's collection coverage is not established by its date range alone.
- District labels include literal None and potentially different geographic levels. Unknown district reports still matter for citywide totals; district rankings need a consistent reviewed geography.
- Categories such as Theft, Traffic-related, Assault, Burglary, Fraud and Drug-related are candidate names, not predetermined model results. Include an Unknown/Unclear option and a documented non-crime/report-type category where needed.
- A topic is a group of similar reports, not automatically a legal crime category. Several topics can map to one category; mixed topics may need a per-report review.
- Use one primary category per report for the main count tables. If the group later adopts multiple labels, document that totals can exceed the report count and rebuild all comparisons consistently.
- The source includes descriptions of harmful events. Keep relevant evidence brief in outputs, avoid unnecessary identifying details, and use report IDs/source URLs for traceability.
- No API calls, paid services, extra scraping or AI-generated embeddings are needed to start. Optional AI-assisted elderly detection needs a separate agreed method and human verification.


## Purpose and dependencies

You prepare the shared dataset that everyone relies on, then return later to analyze district patterns. Errors here affect all four models and every later chart.

**You may start now:** use reports_base. **Your first handover:** prepared data, quality checks, district lookup and duplicate decisions. **Your later dependency:** wait for Member 6's approved report_categories before producing district/category results.

You answer the district-distribution part of Question 1 and the highest-district/composition question (Question 4). Your role has two phases; completing preparation does not complete the district analysis.

### 1. Audit the source

Check rows/columns, missing and blank text, invalid dates, district labels, repeated URLs and repeated title/description combinations. Check every embedding is a finite vector of length 768. Record issues rather than silently changing them. Keep both text fields and both embeddings.

In [6]:
reports = reports_base.copy()
print(reports.shape)
reports.isna().sum()

(5126, 8)


report_id                 0
date_posted               0
title_en                  0
description_en            0
district                  0
desc_en_embeddings768     0
title_en_embeddings768    0
url                       0
dtype: int64

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

### 2. Prepare dates and inclusion flags

Convert date_posted with pd.to_datetime and inspect any failures. Retain every base row/report_id. Add include_in_analysis (boolean) and exclusion_reason. Review duplicate URLs: repeated URL alone is not enough to delete a row. If excluding a confirmed duplicate, retain its row with a reason. Missing district alone should not exclude a report from text modeling or citywide analysis.

In [7]:
# Example: reports["date_posted"] = pd.to_datetime(reports["date_posted"], errors="coerce")
# Inspect invalid dates before agreeing any exclusion rule.

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

### 3. Review district names

Create district_clean using a documented lookup. Treat literal None/blank as Unknown. Agree one geography level: the source may mix neighborhoods and broader districts. Review spelling variants and places outside the agreed study area. Preserve district unchanged. Do not guess locations from the report title. Unknown remains visible in citywide totals.

In [8]:
# TODO: add short code cells for this step.

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

### 4. Publish the preparation handover

Save reports_prepared.parquet in your output folder, plus data_quality.csv, district_lookup.csv and duplicate_review.csv. Validate the report ID set equals the base set and original text/embeddings have not changed. Record the SHA-256 and tell Members 2–4 and 6 that the preparation version is ready. Freeze it before final model runs. Any change requires agreement and reruns.

In [9]:
# Save only after your checks pass:
# reports.to_parquet(output_folder / "reports_prepared.parquet", index=False)

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

### 5. Wait for approved categories, then join them

Load outputs/member_06/report_categories.csv and selected_model.json only after group approval. Join using report_id and validate="one_to_one". Confirm category rows cover every base ID, input checksums match, and eligibility flags are respected. District charts must use the approved category column, not an experimental topic ID.

In [10]:
# TODO: add short code cells for this step.

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

### 6. Count categories by district

Use pandas groupby and size. Export district_category_counts.csv. Calculate share_within_district (category count / all included reports in that district) and share_of_category_reports (district count / all included reports of that category citywide, with Unknown in the denominator). Explain both denominators. Show counts and composition, and include ties in highest_district_by_category.csv.

In [11]:
# Example after the approved join:
# counts = labeled_reports.groupby(["district_clean", "crime_category"]).size()

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

### 7. Map and interpret the distribution

Produce a readable count chart, a district/category heatmap, and a geographic map using suitable boundary data if available. The source has no coordinates: a scatterplot is not a district map. Record the boundary source/license and unmatched labels. Coordinate with the group to obtain compatible official boundaries; raise missing boundary access before final submission. Never label a count hotspot as a risk rate without population/exposure data.

In [12]:
# TODO: add short code cells for this step.

**Explain your output:** What did you observe? Why does it matter? What remains uncertain?

**What to be careful about:** A district with many reports may have more reporting, activity or residents. Explain observed counts; do not recommend patrol allocations as if this dataset measures complete crime incidence.

## What you must submit to the main workstation

Submit this completed notebook with saved outputs, your files in outputs/member_01, and figures in figures/member_01. Run the notebook top-to-bottom before submission. Do not submit empty templates as results.

- **data_quality.csv** = columns: `check`, `value`, `notes`.
- **district_lookup.csv** = columns: `district_original`, `district_clean`, `decision`, `notes`.
- **duplicate_review.csv** = columns: `report_id`, `url`, `decision`, `reason`.
- **district_category_counts.csv** = columns: `district_clean`, `crime_category`, `report_count`, `share_within_district`, `share_of_category_reports`.
- **highest_district_by_category.csv** = columns: `crime_category`, `district_clean`, `report_count`.
- **run_metadata.json** = name, input checksums, package versions, method settings, run date, record counts and status.
- **handover.md** = what you did, results, limitations, reviewer comments, output inventory and outstanding questions.
- **reports_prepared.parquet** = all base IDs/original columns, with datetime date_posted, district_clean, include_in_analysis and exclusion_reason.

**Main notebook destination:** Data preparation; District distribution and citywide composition.

Submit preparation files first so the group can work in parallel. Submit the district tables/maps after approved categories are available. Keep ties in the highest-district table and show Unknown separately.
### Output conventions

- Keep report_id as the original BR string. Topic IDs are integers within a model, or empty for missing/excluded inputs; HDBSCAN noise is -1.
- For model tables, use the exact model_id and assignment_status values: assigned, unassigned, excluded, missing_text, invalid_embedding. Keywords can be semicolon-separated text. n_reports is the number of reports represented by that topic/status, not a score.
- Store shares/agreement rates as fractions from 0 to 1; multiply by 100 only when displaying percentages. Record which report set is the denominator. A model weight is not a calibrated category probability.
- Date keys use YYYY-MM-DD; monthly keys use YYYY-MM. Keep literal text labels intact when reading lookup CSVs (keep_default_na=False can preserve the original text None).
- Copy templates/member_XX/run_metadata.json into your output folder only when filling it with an actual run. Keep the common metadata keys, replace placeholders, and mark submitted/reviewed only when true. Record SHA-256 file versions, not just filenames.
- Final category statuses are mapped_topic/manual_override/unclear/unassigned/excluded/missing_text/invalid_embedding. Elderly review statuses are yes/no_evidence/unclear/not_reviewed/excluded. Unclear and unreviewed are not negative results.

### Final submission checks

- [ ] All expected output files are populated, or a genuine empty result is explained (for example, no anomalies met the rule).
- [ ] Every report-level table uses report_id and includes the required cohort, without unexpected duplicates.
- [ ] IDs, source/prepared checksums, categories and denominators match the shared contract.
- [ ] Required charts have descriptive titles, labels, units, periods and readable text; export PNG/HTML as appropriate.
- [ ] Code runs in order; results and explanations agree; raw/debug output is bounded.
- [ ] A peer reviewer has checked the method, sample reports, files and conclusions.
- [ ] handover.md says exactly what to import into the main notebook and lists anything still unresolved.

### Your handover summary

**What I completed:** ______  
**Most useful finding (after execution):** ______  
**Output files and row counts:** ______  
**Main limitations:** ______  
**Reviewer decision:** ______  
**Next person who needs my outputs:** ______
